In [7]:
import random

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from sklearn.metrics import cohen_kappa_score
from torchvision import transforms
from tqdm import tqdm

from src.config import IMG_DIR, SPLITS_DIR, PROCESSED_DIR
from src.data.dataset import DiabeticRetinopathyDataset, get_dataloader

In [27]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

### Experiment & Parameters
Verifying the functionality of the complete training pipeline (data loading -> training loop with validation) on a reduced local subset.

- **Architecture:** efficientnet_lite0 (pretrained)
- **Data subsets:** 200 train images, 50 validation images
- **Hyperparameters:** Learning Rate = 1e-4, Batch Size = 16, Epochs = 3, Seed = 42

In [28]:
train_df = pd.read_csv(SPLITS_DIR / "train_split.csv")[:200]
val_df = pd.read_csv(SPLITS_DIR / "val_split.csv")[:50]

print("Counts of diagnosis categories for train and val subsets:")
print(train_df["diagnosis"].value_counts().sort_index())
print(val_df["diagnosis"].value_counts().sort_index())

Counts of diagnosis categories for train and val subsets:
diagnosis
0    91
1    20
2    58
3    15
4    16
Name: count, dtype: int64
diagnosis
0    25
1     7
2    10
3     2
4     6
Name: count, dtype: int64


In [29]:
train_dataset = DiabeticRetinopathyDataset(
    img_dir=str(IMG_DIR),
    annotations_df=train_df,
    split="train",
    preprocessed_dir=PROCESSED_DIR / "images_processed",
    transform=transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
)
val_dataset = DiabeticRetinopathyDataset(
    img_dir=str(IMG_DIR),
    annotations_df=val_df,
    split="val",
    preprocessed_dir=PROCESSED_DIR / "images_processed",
    transform=transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
)

train_loader = get_dataloader(train_dataset, batch_size=16, num_workers=2)
val_loader = get_dataloader(val_dataset, batch_size=16, num_workers=2)

In [30]:
class BaselinePretrainedModel(nn.Module):
    def __init__(self, model_name='efficientnet_lite0', num_classes=5):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

In [31]:
batch = next(iter(train_loader))
print(f"Batch shape: {batch['image'].shape}")
print(f"Batch labels: {batch['label']}")
logits = (BaselinePretrainedModel()(batch["image"]))
print("Model outputs:")
print(logits[:5])

Batch shape: torch.Size([16, 3, 384, 384])
Batch labels: tensor([0, 3, 4, 2, 0, 3, 0, 1, 1, 0, 0, 1, 0, 0, 0, 2])
Model outputs:
tensor([[-2.1825,  0.1039,  2.6183,  0.1777, -0.0749],
        [-1.2251,  3.6935,  2.0948,  0.5329,  3.0895],
        [ 0.1126,  0.9102,  0.4324,  2.4663,  1.8728],
        [-2.2006,  1.7889,  1.8016,  0.8186,  0.4010],
        [-0.0644, -0.0746, -3.1065,  1.5114,  0.7167]],
       grad_fn=<SliceBackward0>)


In [32]:
set_seed()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = BaselinePretrainedModel()
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

num_epochs = 3
for epoch in range(num_epochs):
    # Train
    model.train()
    train_loss = 0.0
    total_samples = 0

    with tqdm(train_loader, desc=f"Epoch [Train {epoch+1}/{num_epochs}]", leave=False) as pbar:
        for batch in pbar:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            out = model(images)
            loss = loss_fn(out, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            current_batch_size = images.shape[0]
            total_samples += current_batch_size
            train_loss += loss.item() * current_batch_size

    avg_train_loss = train_loss / total_samples

    # Evaluate
    model.eval()
    val_loss = 0.0
    total_samples = 0
    all_preds = []
    all_targets = []

    with tqdm(val_loader, desc=f"Epoch [Val {epoch+1}/{num_epochs}]", leave=False) as pbar:
        with torch.no_grad():
            for batch in pbar:
                images = batch["image"].to(device)
                labels = batch["label"].to(device)

                out = model(images)
                loss = loss_fn(out, labels)
                current_batch_size = images.shape[0]
                total_samples += current_batch_size
                val_loss += loss.item() * current_batch_size

                preds = torch.argmax(out, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_samples

    val_qwk = cohen_kappa_score(all_targets, all_preds, weights='quadratic')
    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val QWK: {val_qwk:.4f}")

Device: cpu


Epoch [1/3] | Train Loss: 1.6698 | Val Loss: 1.8203 | Val QWK: 0.7007


Epoch [2/3] | Train Loss: 0.3057 | Val Loss: 1.0647 | Val QWK: 0.6281


Epoch [3/3] | Train Loss: 0.0820 | Val Loss: 1.0478 | Val QWK: 0.6854


What was verified:
* Data loaders: Successfully batched and loaded image tensors into the model.
* Gradient flow and optimizer updates: Training loss decreased from 1.67 to 0.08 over 3 epochs, consistent with proper backpropagation.
* Validation: Successfully executed validation loop.